In [ ]:
import os
import joblib
import pandas as pd
import matplotlib.pyplot as plt 

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


df = pd.read_csv("../data/processed/cleaned_data.csv")
df = df[df["km_driven"] != 3800000].copy()

features = [
    "vehicle_age",
    "km_driven",
    "mileage",
    "engine",
    "max_power",
    "seats"
]

X = df[features]
# print(X)

scaler = StandardScaler()
x_scaled = scaler.fit_transform(X) 


inertia = []

for k in range(2,11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(x_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(10,5))

plt.plot(range(2,11), inertia, marker='o')
plt.ylabel("Inertia")
plt.title("Elbow Method")

# plt.show()

# k = 5

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
pred = kmeans.fit_predict(x_scaled)

df["cluster"] = pred
cluster_counts = df["cluster"].value_counts().sort_index()
print(cluster_counts)


pca = PCA(n_components=2)
X_pca = pca.fit_transform(x_scaled)
plt.figure(figsize=(8, 6))
plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=df["cluster"],
    s=10
)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("Vehicle Clusters")
plt.show()


os.makedirs("../models/clustering", exist_ok=True)
joblib.dump(kmeans,"../models/clustering/kmeans_model.pkl")
joblib.dump(scaler,"../models/clustering/scaler.pkl")

df.to_csv("../models/clustering/clustered_data.csv",index=False)